# LongFlow P1 -- Capture v2 gate check (hard constraint 6: never skip this)

Runtime: **L4 GPU** is plenty (no batched generation here, just training +
a handful of decodes). ~20-30 min, ~$1.

Capture v2 is DONE (248 scripts, 510,143 frame pairs, `capture_v2_manifest.json`)
-- this notebook is the mandatory gate before that cache feeds any real
training spend: roundtrip sanity, a 5K-step head trained on the FULL v2
cache (no need to sub-sample down to literal "1K" -- capture is already
paid for, training 5K steps is cheap regardless of pool size), then
decode + listen across BOTH short and long-context scripts, since
context-length coverage is the entire reason capture v2 exists.

Pre-registered: `experiments/p1_flow_head/NOTES.md`
("CAPTURE V2 COMPLETE" entry). Writes checkpoints to a **new** tag
(`gate_v2_5k.pt`) so nothing overwrites the original P1 gate checkpoint.

In [ ]:
# ===== COLD START (idempotent) -- run me first, wait for READY =====
NOTEBOOK_VERSION = "Capture v2 gate check v1.0 (2026-08-16)"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, sys, time
CACHE_DIR = "/content/drive/MyDrive/longflow_p1_cache_v2"  # capture v2's cache, NOT v1's
CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)
assert os.path.exists(CACHE_DIR), f"{CACHE_DIR} not found -- capture v2 must be run first"

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone failed -- check repo access"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.cache.capture import load_utterance
from src.flow_head.model import FlowHead, FlowHeadConfig
from src.flow_head.trainer import load_pairs, save_checkpoint, load_checkpoint, train, sample_latents
print("READY")

In [ ]:
# ===== Cache summary -- confirm dual-stream + sigma fields survived intact =====
files = sorted(glob.glob(f"{CACHE_DIR}/*.pt"))
print(f"{len(files)} cached scripts")

n_neg, n_sigma, total_frames = 0, 0, 0
bins = {}
for f in files:
    utt = load_utterance(f)  # runs assert_frame_aligned on read -- cheap corruption check
    total_frames += utt.hidden.shape[0]
    if utt.neg_hidden is not None:
        n_neg += 1
    if utt.sigma is not None:
        n_sigma += 1
    tw = utt.meta.get("target_words", "?")
    bins[tw] = bins.get(tw, 0) + 1

print(f"total frames: {total_frames}")
print(f"scripts with neg_hidden: {n_neg}/{len(files)}")
print(f"scripts with sigma: {n_sigma}/{len(files)}")
print("word-bin counts:", bins)
assert n_neg == len(files), "dual-stream capture incomplete -- some scripts missing neg_hidden"
assert n_sigma == len(files), "sigma capture incomplete -- some scripts missing sigma track"
print("PASS: every cached script carries both v2 fields, no read-time alignment errors")

In [ ]:
# ===== 1. Roundtrip sanity -- gate criterion (a), do NOT skip =====
# Mirrors generate()'s un-scaling then decode -- verify against source if VibeVoice
# ever updates (grep speech_scaling_factor / acoustic_tokenizer.decode / speech_bias
# in modeling_vibevoice_inference.py, same check the original P1 gate ran).
import soundfile as sf
from IPython.display import Audio, display

def decode_latents(z):  # z: [T, d_latent] head-space, fp
    sc = model.model.speech_scaling_factor
    bi = model.model.speech_bias_factor
    z = z.to("cuda", torch.bfloat16)
    z = z / sc - bi
    for shape in (z.unsqueeze(0), z.unsqueeze(0).transpose(1, 2)):
        try:
            out = model.model.acoustic_tokenizer.decode(shape)
            wav = out[0] if isinstance(out, tuple) else out
            return wav.detach().float().cpu().numpy().squeeze()
        except Exception as e:
            print(f"decode attempt {tuple(shape.shape)} failed: {repr(e)[:150]}")
    raise RuntimeError("both decode shapes failed -- paste the errors to Claude")

os.makedirs("/content/gate_v2_audio", exist_ok=True)
d0 = load_utterance(files[0])
print(d0.utt_id, "|", d0.text[:100], "| frames:", d0.hidden.shape[0])
wav = decode_latents(d0.latent.float())
sf.write("/content/gate_v2_audio/roundtrip.wav", wav, 24000)
display(Audio("/content/gate_v2_audio/roundtrip.wav"))
print("LISTEN: does this sound like normal VibeVoice speech saying the text above?")

In [ ]:
# ===== 2. Train the flow head -- 5K steps on the FULL v2 cache =====
# No sub-sampling to a literal "1K": capture is already paid for, and 5K
# steps is cheap regardless of pool size (unlike a real 75K-scale run).
!mkdir -p /content/cache_local && cp {CACHE_DIR}/*.pt /content/cache_local/

data = load_pairs("/content/cache_local")
print(f"pairs: {data.hidden.shape[0]}  d_model={data.d_model}  d_latent={data.d_latent}")
head = FlowHead(FlowHeadConfig(d_model=data.d_model, d_latent=data.d_latent))
print(f"head params: {head.param_count()/1e6:.2f}M")
out = train(head, data, steps=5000, batch_size=1024, lr=2e-4,
           ema_decay=0.999, device="cuda", log_every=500)
save_checkpoint(f"{CKPT_DIR}/gate_v2_5k.pt", head, out["ema"], data, step=5000)
print("checkpoint saved to Drive -- gate_v2_5k.pt (does not overwrite the v1 gate ckpt)")

In [ ]:
# ===== 3. Decode flow-head samples -- gate criterion (c), LISTEN =====
# Sample across BOTH short (150w) and long (2400w) bins -- context-length
# coverage is the whole point of capture v2, so the gate should check it,
# not just spot-check whichever files sort first/last like the v1 gate did.
head_ema, lat_mean, lat_std = load_checkpoint(f"{CKPT_DIR}/gate_v2_5k.pt")
head_ema = head_ema.to("cuda")

short_files = sorted(glob.glob("/content/cache_local/cv2_150w_*.pt"))[:2]
long_files = sorted(glob.glob("/content/cache_local/cv2_2400w_*.pt"))[:2]

for f in short_files + long_files:
    utt = load_utterance(f)
    tag = utt.utt_id
    for nfe in (4, 16):
        z = sample_latents(head_ema, utt.hidden.float(), lat_mean, lat_std, nfe=nfe)
        sf.write(f"/content/gate_v2_audio/{tag}_flow{nfe}.wav", decode_latents(z), 24000)
    sf.write(f"/content/gate_v2_audio/{tag}_teacher.wav", decode_latents(utt.latent.float()), 24000)
    print(tag, f"({utt.hidden.shape[0]} frames) |", utt.text[:70])
    for suffix in ("teacher", "flow4", "flow16"):
        print(" ", suffix)
        display(Audio(f"/content/gate_v2_audio/{tag}_{suffix}.wav"))

In [ ]:
# ===== Bundle for download =====
import zipfile
with zipfile.ZipFile("/content/gate_v2_audio.zip", "w") as z:
    for f in os.listdir("/content/gate_v2_audio"):
        z.write(f"/content/gate_v2_audio/{f}", f)
from google.colab import files as colab_files
colab_files.download("/content/gate_v2_audio.zip")